# Шаблон: выгрузка всех майских транзакций по 2 agr_id в один Excel

Этот мини-ноутбук нужен как донор кода для уже прогруженной рабочей тетрадки.

Что делает:
- автоматически отбирает 2 `agr_id` по строгим фильтрам:
  - один тариф `Стандартный`, один `Индивидуальный`;
  - `trx_cnt` совпадает между Lake и Excel;
  - `trx_cnt <= 15`;
  - `Excel > Lake` по `commission_total`, `int_component`, `commission_monthly`;
- выгружает **все** транзакции за выбранный месяц по этим 2 `agr_id`;
- сохраняет один Excel-файл с 2 листами (по одному `agr_id` на лист).

Ожидается, что в сессии уже есть подключение `imp` к Impala и DataFrame `final_df`.

In [ ]:
from pathlib import Path
import re
from decimal import Decimal, InvalidOperation
import pandas as pd

# Параметры
report_month = '2026-05-01'
output_trx_xlsx_path = './may_trx_two_agr.xlsx'
excel_path = '/home/jovyan/documents/Equaring/Data/05_Май_2026.xlsx'
excel_header = 0
max_trx_cnt = 15

# Вариант 1: вручную указать agr_id (если пусто, будет авто-отбор по фильтрам)
selected_agr_ids = [
    # '123456789',
    # '987654321',
]


def normalize_inn_q1(value):
    if pd.isna(value):
        return None
    s = str(value).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    if len(s) not in (10, 12):
        return None
    return s


def normalize_agr_q1(value):
    if pd.isna(value):
        return None
    s = str(value).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    if s in {'', 'nan', 'None'}:
        return None
    try:
        d = Decimal(s)
        if d == d.to_integral_value():
            return str(int(d))
    except (InvalidOperation, ValueError):
        pass
    s = re.sub(r'\.0$', '', s)
    if s in {'', 'nan', 'None'}:
        return None
    return s


def pick_col_robust(columns, candidates):
    cols = list(columns)
    norm = lambda x: re.sub(r'\s+', ' ', str(x).replace('\xa0', ' ').strip().lower())
    nmap = {norm(c): c for c in cols}
    for c in candidates:
        if c in cols:
            return c
        nc = norm(c)
        if nc in nmap:
            return nmap[nc]
    return None


def to_num_series(s):
    return pd.to_numeric(
        s.astype(str)
         .str.replace('\xa0', '', regex=False)
         .str.replace(' ', '', regex=False)
         .str.replace(',', '.', regex=False),
        errors='coerce'
    )


report_month_ts = pd.to_datetime(report_month)
month_start = report_month_ts.strftime('%Y-%m-%d')
month_end = (report_month_ts + pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')

# Авто-отбор agr_id по строгим фильтрам, если вручную не задано
if not selected_agr_ids:
    if 'final_df' not in globals() or final_df is None or len(final_df) == 0:
        raise RuntimeError('Не найден final_df. Для авто-отбора сначала сформируйте final_df в основной тетрадке.')

    ex = pd.read_excel(excel_path, header=excel_header)
    col_map = {
        'inn_col': ['ИНН', 'inn', 'c_inn'],
        'agr_col': ['ID договора', 'Номер договора', 'agr_id', 'abs_agr_id'],
        'trx_cnt_col': ['Количество операций', 'Количеств операций', 'trx_cnt'],
        'comm_monthly_col': ['Комиссия CN (₽ в месяц)', 'Комиссия (₽ в месяц)', 'Комиссия \n(₽ в месяц)', 'Комиссия (руб в месяц)'],
        'comm_total_col': ['Комиссия эквайринга'],
        'irf_col': ['Комиссия МПС (IRF, ₽)', 'Комиссия МПС (IRF, р)', 'Комиссия МПС (IRF, руб)', 'Комиссия МПС (IRF)'],
    }
    resolved = {k: pick_col_robust(ex.columns, v) for k, v in col_map.items()}
    missing = [k for k, v in resolved.items() if v is None]
    if missing:
        raise ValueError(f'Не найдены колонки Excel: {missing}. Доступные: {list(ex.columns)}')

    ex['inn_key'] = ex[resolved['inn_col']].apply(normalize_inn_q1)
    ex['agr_id_key'] = ex[resolved['agr_col']].apply(normalize_agr_q1)
    ex['trx_cnt_excel'] = pd.to_numeric(ex[resolved['trx_cnt_col']], errors='coerce')
    ex['commission_monthly_excel'] = to_num_series(ex[resolved['comm_monthly_col']])
    ex['commission_total_excel'] = to_num_series(ex[resolved['comm_total_col']])
    ex['int_component_excel'] = to_num_series(ex[resolved['irf_col']])

    ex_agg = (
        ex.dropna(subset=['inn_key', 'agr_id_key'])
          .groupby(['inn_key', 'agr_id_key'], as_index=False)
          .agg({
              'trx_cnt_excel': 'max',
              'commission_monthly_excel': 'max',
              'commission_total_excel': 'sum',
              'int_component_excel': 'sum',
          })
    )

    lk = final_df.copy()
    lk['inn_key'] = lk['inn'].apply(normalize_inn_q1)
    lk['agr_id_key'] = lk['agr_id'].apply(normalize_agr_q1)
    lk['trx_cnt_lake'] = pd.to_numeric(lk['trx_cnt'], errors='coerce')
    lk['commission_monthly_lake'] = pd.to_numeric(lk['commission_monthly'], errors='coerce')
    lk['commission_from_ops_lake'] = pd.to_numeric(lk.get('commission_from_ops'), errors='coerce').fillna(0)
    lk['commission_total_lake'] = lk['commission_from_ops_lake'] + lk['commission_monthly_lake']
    lk['int_component_lake'] = pd.to_numeric(lk['int_component'], errors='coerce')
    lk['tariff_name'] = lk.get('tariff_name')

    lk_agg = (
        lk.dropna(subset=['inn_key', 'agr_id_key'])
          .groupby(['inn_key', 'agr_id_key'], as_index=False)
          .agg({
              'trx_cnt_lake': 'max',
              'commission_monthly_lake': 'max',
              'commission_total_lake': 'max',
              'int_component_lake': 'max',
              'tariff_name': 'first',
          })
    )

    cmp = lk_agg.merge(ex_agg, on=['inn_key', 'agr_id_key'], how='inner')
    cmp['trx_cnt_delta'] = cmp['trx_cnt_lake'].fillna(0) - cmp['trx_cnt_excel'].fillna(0)
    cmp['commission_total_delta'] = cmp['commission_total_lake'].fillna(0) - cmp['commission_total_excel'].fillna(0)
    cmp['int_component_delta'] = cmp['int_component_lake'].fillna(0) - cmp['int_component_excel'].fillna(0)
    cmp['commission_monthly_delta'] = cmp['commission_monthly_lake'].fillna(0) - cmp['commission_monthly_excel'].fillna(0)

    cand = cmp[
        (cmp['trx_cnt_delta'] == 0) &
        (cmp['trx_cnt_lake'].fillna(cmp['trx_cnt_excel']).between(1, max_trx_cnt, inclusive='both')) &
        (cmp['commission_total_delta'] < 0) &
        (cmp['int_component_delta'] < 0) &
        (cmp['commission_monthly_delta'] < 0)
    ].copy()

    cand['score'] = (
        cand['commission_total_delta'].abs() +
        cand['int_component_delta'].abs() +
        cand['commission_monthly_delta'].abs()
    )

    cand['tariff_name_norm'] = cand['tariff_name'].astype(str).str.lower()
    std_top = cand[cand['tariff_name_norm'].str.contains('стандарт', na=False)].sort_values('score', ascending=False).head(3)
    ind_top = cand[cand['tariff_name_norm'].str.contains('индивиду', na=False)].sort_values('score', ascending=False).head(3)

    cases_main = pd.concat([
        std_top.head(1).assign(case_group='standard_main'),
        ind_top.head(1).assign(case_group='individual_main')
    ], ignore_index=True)

    cases_reserve = pd.concat([
        std_top.iloc[1:].assign(case_group='standard_reserve') if len(std_top) > 1 else pd.DataFrame(),
        ind_top.iloc[1:].assign(case_group='individual_reserve') if len(ind_top) > 1 else pd.DataFrame(),
    ], ignore_index=True)

    selected_agr_ids = cases_main['agr_id_key'].dropna().astype(str).tolist()
    selected_agr_ids = list(dict.fromkeys([x for x in selected_agr_ids if x]))

selected_agr_ids = [str(x).strip() for x in selected_agr_ids if str(x).strip()]
selected_agr_ids = list(dict.fromkeys(selected_agr_ids))

if len(selected_agr_ids) != 2:
    raise ValueError('Нужно ровно 2 agr_id после отбора. Проверьте фильтры или задайте selected_agr_ids вручную.')

print('Выбранные agr_id:', selected_agr_ids)
print('Период выгрузки:', month_start, '->', month_end)

if 'cases_main' in globals() and len(cases_main) == 2:
    print('Основные кейсы по фильтрам:')
    display(cases_main[['case_group', 'agr_id_key', 'tariff_name', 'trx_cnt_lake', 'trx_cnt_excel', 'commission_total_delta', 'int_component_delta', 'commission_monthly_delta', 'score']])
    if 'cases_reserve' in globals() and len(cases_reserve):
        print('Резервные кейсы:')
        display(cases_reserve[['case_group', 'agr_id_key', 'tariff_name', 'trx_cnt_lake', 'trx_cnt_excel', 'commission_total_delta', 'int_component_delta', 'commission_monthly_delta', 'score']])

In [ ]:
if 'imp' not in globals() or imp is None:
    raise RuntimeError('Не найдено подключение imp. Сначала выполните ячейку подключения к Impala.')

agr_in = ', '.join([f"'{x}'" for x in selected_agr_ids])

sql_two_agr_trx = f"""
with sa_agr as (
  select distinct
    cast(a.abs_agr_id as string) as agr_id,
    cast(a.n_agr as string) as n_agr
  from ods_alpha.scd1_agreements a
  where cast(a.abs_agr_id as string) in ({agr_in})
),
trx_base_raw as (
  select
    cast(t.n_trx as string) as n_trx,
    cast(t.c_nter as string) as c_nter,
    cast(t.n_amt_src as double) as n_amt_src,
    cast(to_date(cast(t.d_trx_orig as timestamp)) as string) as trx_date
  from ods_alpha.scd1_trx t
  left join ods_alpha.scd1_base24_fiids fa
    on cast(fa.c_fiid as string) = cast(t.c_fiid_acq as string)
  where to_date(cast(t.d_trx_orig as timestamp)) between cast('{month_start}' as date) and cast('{month_end}' as date)
    and t.c_nter is not null
    and coalesce(t.ods_deleted_flg, '0') <> '1'
    and t.c_trx_class = 'SA'
    and t.c_trx_type = 'S01'
    and coalesce(t.cf_trx_stat, '') <> 'R'
    and coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
),
trx_base as (
  select
    n_trx,
    max(c_nter) as c_nter,
    max(n_amt_src) as n_amt_src,
    max(trx_date) as trx_date
  from trx_base_raw
  group by n_trx
),
ta_raw as (
  select
    cast(a.n_trx as string) as n_trx,
    cast(a.n_agr as string) as n_agr,
    coalesce(cast(a.n_amt_tax as double), 0.0) as n_amt_tax
  from ods_alpha.scd1_trx_acq a
  join sa_agr s
    on s.n_agr = cast(a.n_agr as string)
  join trx_base tb
    on tb.n_trx = cast(a.n_trx as string)
),
ta as (
  select
    n_trx,
    n_agr,
    max(n_amt_tax) as n_amt_tax
  from ta_raw
  group by n_trx, n_agr
),
trx_keys as (
  select distinct n_trx
  from ta
),
trx_int_agg as (
  select
    cast(i.n_trx as string) as n_trx,
    sum(coalesce(cast(i.n_amt_fee as double), 0.0)) as n_amt_fee
  from ods_alpha.scd1_trx_int i
  join trx_keys k
    on k.n_trx = cast(i.n_trx as string)
  group by cast(i.n_trx as string)
)
select
  s.agr_id,
  ta.n_agr,
  ta.n_trx,
  tb.trx_date,
  tb.c_nter,
  tb.n_amt_src as trx_sum,
  ta.n_amt_tax as commission_from_ops,
  coalesce(i.n_amt_fee, 0.0) as int_component
from ta
join sa_agr s
  on s.n_agr = ta.n_agr
join trx_base tb
  on tb.n_trx = ta.n_trx
left join trx_int_agg i
  on i.n_trx = ta.n_trx
order by s.agr_id, ta.n_trx
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    two_agr_trx_df = imp.fetch(sql_two_agr_trx)

if two_agr_trx_df is None:
    two_agr_trx_df = pd.DataFrame(columns=[
        'agr_id', 'n_agr', 'n_trx', 'trx_date', 'c_nter', 'trx_sum', 'commission_from_ops', 'int_component'
    ])

for c in ['agr_id', 'n_agr', 'n_trx', 'trx_date', 'c_nter']:
    if c in two_agr_trx_df.columns:
        two_agr_trx_df[c] = two_agr_trx_df[c].astype(str)

print(f'Выгружено строк trx: {len(two_agr_trx_df):,}')
display(two_agr_trx_df.head(20))

In [ ]:
def safe_sheet_name(agr_id):
    name = f'agr_{agr_id}'
    name = re.sub(r"[\\/*?:\[\]]", '_', name)
    return name[:31]

out_path = Path(output_trx_xlsx_path)
out_path.parent.mkdir(parents=True, exist_ok=True)

sheet_stats = []
with pd.ExcelWriter(out_path, engine='xlsxwriter') as writer:
    for agr_id in selected_agr_ids:
        df_sheet = two_agr_trx_df[two_agr_trx_df['agr_id'] == str(agr_id)].copy()
        sheet_name = safe_sheet_name(agr_id)

        if df_sheet.empty:
            pd.DataFrame(columns=two_agr_trx_df.columns).to_excel(writer, sheet_name=sheet_name, index=False)
        else:
            df_sheet.to_excel(writer, sheet_name=sheet_name, index=False)

        sheet_stats.append({
            'agr_id': str(agr_id),
            'sheet_name': sheet_name,
            'rows': len(df_sheet),
            'unique_agr_in_sheet': int(df_sheet['agr_id'].nunique()) if len(df_sheet) else 0,
        })

sheet_stats_df = pd.DataFrame(sheet_stats)
print(f'Excel сохранен: {out_path.resolve()}')
display(sheet_stats_df)

if len(sheet_stats_df) != 2:
    print('WARN: ожидалось 2 листа, проверьте selected_agr_ids.')

if (sheet_stats_df['unique_agr_in_sheet'] > 1).any():
    print('WARN: в одном из листов найдено более одного agr_id.')

if (sheet_stats_df['rows'] == 0).any():
    print('WARN: для одного из agr_id нет строк по заданным фильтрам.')

# Дополнительные проверки строгих условий отбора
if 'cases_main' in globals() and len(cases_main) == 2:
    strict_ok = (
        (pd.to_numeric(cases_main['trx_cnt_delta'], errors='coerce').fillna(999999) == 0).all() and
        (pd.to_numeric(cases_main['trx_cnt_lake'], errors='coerce').fillna(pd.to_numeric(cases_main['trx_cnt_excel'], errors='coerce')).between(1, max_trx_cnt, inclusive='both')).all() and
        (pd.to_numeric(cases_main['commission_total_delta'], errors='coerce').fillna(0) < 0).all() and
        (pd.to_numeric(cases_main['int_component_delta'], errors='coerce').fillna(0) < 0).all() and
        (pd.to_numeric(cases_main['commission_monthly_delta'], errors='coerce').fillna(0) < 0).all()
    )

    has_std = cases_main['tariff_name'].astype(str).str.lower().str.contains('стандарт', na=False).any()
    has_ind = cases_main['tariff_name'].astype(str).str.lower().str.contains('индивиду', na=False).any()

    print(f'Проверка строгих условий: {"OK" if strict_ok else "WARN"}')
    print(f'Покрытие тарифов (Стандартный/Индивидуальный): {"OK" if (has_std and has_ind) else "WARN"}')

    if not strict_ok:
        print('WARN: один или оба кейса не удовлетворяют строгим условиям. Проверьте cases_main/cases_reserve.')
    if not (has_std and has_ind):
        print('WARN: не найдено покрытие по двум тарифам. Проверьте фильтры или используйте резервные кейсы.')
else:
    print('WARN: cases_main не найден или содержит не 2 строки. Для строгой валидации сначала выполните авто-отбор.')